# 11_v5c_offense_tilt — V5C 3.1 进攻层加杠杆测试

> 测试: 从 VOO 拿 5% 给 QQQ (V5C 3.1f) 或 TQQQ (V5C 3.1g), 其他层不变

## 配置对比

| | V5C 3.1 (基准) | V5C 3.1f (加 QQQ) | V5C 3.1g (加 TQQQ) |
|---|---|---|---|
| VOO | 15% | **10%** | **10%** |
| QQQ | 15% | **20%** | 15% |
| TQQQ | — | — | **5%** |
| 其他 | 不变 | 不变 | 不变 |

## 数据窗口

- TQQQ: 2010-02 上市 → 16Y 窗口
- 其他: 23.8Y 可用

**两个测试**:
1. **23.8Y**: V5C 3.1 vs V5C 3.1f (无 TQQQ)
2. **16Y**: V5C 3.1 vs V5C 3.1f vs V5C 3.1g (含 TQQQ)

## 期望发现

- **科技牛市 (2020-2021)**: 3.1f/g 应大幅领先
- **2022 Bear**: 3.1f/g 应较差 (NDX -33% vs SPX -25%)
- **2008 GFC**: NDX 跌幅与 SPX 相近, 应差距小
- **整体 Sharpe**: 看牛市增益是否补偿熊市加深

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

tickers = ['VFINX','QQQ','TQQQ','HQH','XLV','VFITX','GLD','GC=F','DBC','PCRIX']
raw = yf.download(tickers, start='1999-01-01', auto_adjust=True)['Close']

print('数据起始:')
for t in tickers:
    if t in raw.columns:
        first = raw[t].first_valid_index()
        print(f'  {t:<8}: {first.date() if first else "N/A"}')

In [ ]:
def synthesize(short, long):
    short = short.dropna(); long = long.dropna()
    if len(short) == 0: return long
    overlap = short.index[0]
    if overlap <= long.index[0]: return short
    long_at = long.loc[:overlap].iloc[-1] if not long.loc[:overlap].empty else long.iloc[0]
    scale = short.iloc[0] / long_at
    early = long.loc[:overlap].iloc[:-1] * scale
    return pd.concat([early, short]).sort_index().pipe(lambda s: s[~s.index.duplicated(keep='last')])

data = pd.DataFrame({
    'VOO': raw['VFINX'], 'QQQ': raw['QQQ'], 'TQQQ': raw['TQQQ'],
    'HQH': raw['HQH'], 'XLV': raw['XLV'],
    'VGSH': raw['VFITX'],
    'GLDM': synthesize(raw['GLD'], raw['GC=F']),
    'BCX': synthesize(raw['DBC'], raw['PCRIX']),
})
for c in data.columns:
    fv = data[c].first_valid_index()
    print(f'  {c:<6}: {fv.date() if fv else "N/A"}')

In [ ]:
def simulate_rebalance(returns_df, target_weights, threshold_pp=5.0):
    used = [t for t in target_weights.keys() if t in returns_df.columns]
    sub = returns_df[used].dropna()
    target = np.array([target_weights[t] for t in used]); target = target/target.sum()
    cw = target.copy(); pr=[]; rd=[sub.index[0]]
    for date, dr in sub.iterrows():
        pr.append(np.sum(cw*dr.values))
        nw = cw*(1+dr.values); nw = nw/nw.sum()
        if np.max(np.abs(nw-target))*100 >= threshold_pp:
            cw = target.copy(); rd.append(date)
        else:
            cw = nw
    return pd.Series(pr, index=sub.index), rd

def metrics(rs, dates, name):
    cum = (1+rs).cumprod()
    n_y = len(rs)/252
    cagr = cum.iloc[-1]**(1/n_y) - 1
    vol = rs.std()*np.sqrt(252)
    sharpe = (cagr-0.04)/vol
    downside = rs[rs<0]
    sortino = (cagr-0.04)/(downside.std()*np.sqrt(252))
    rm = cum.expanding().max()
    dd = (cum/rm - 1)
    return {'Name':name,'CAGR':cagr,'Vol':vol,'Sharpe':sharpe,'Sortino':sortino,
            'Max DD':dd.min(),'Max DD Date':dd.idxmin(),
            'Calmar':cagr/abs(dd.min()),'Rebalances':len(dates)-1}

def show_compare(metric_list):
    cols = ['CAGR','Vol','Sharpe','Sortino','Max DD','Calmar']
    print(f"{'Metric':<10}", end='')
    for m in metric_list: print(f"  {m['Name']:<22}", end='')
    print()
    print('-'*90)
    for col in cols:
        fmt = '{:.2%}' if col not in ['Sharpe','Sortino','Calmar'] else '{:.3f}'
        line = f'{col:<10}'
        for m in metric_list:
            line += f'  {fmt.format(m[col]):<22}'
        print(line)
    print(f"{'Rebalances':<10}", end='')
    for m in metric_list: print(f"  {str(m['Rebalances']):<22}", end='')
    print()

In [ ]:
# ============================================================
# 测试 1: 23.8Y - V5C 3.1 vs V5C 3.1f
# ============================================================
print('=' * 90)
print('测试 1: 23.8Y - V5C 3.1 vs V5C 3.1f (VOO → QQQ)')
print('=' * 90)

data_long = data[['VOO','QQQ','HQH','XLV','VGSH','GLDM','BCX']].dropna()
ret_long = data_long.pct_change().dropna()
print(f'窗口: {data_long.index[0].date()} → {data_long.index[-1].date()} ({len(data_long)/252:.1f} 年)')

V5C_3_1 =  {'VOO':0.15,'QQQ':0.15,'HQH':0.10,'XLV':0.10,'GLDM':0.20,'BCX':0.10,'VGSH':0.20}
V5C_3_1f = {'VOO':0.10,'QQQ':0.20,'HQH':0.10,'XLV':0.10,'GLDM':0.20,'BCX':0.10,'VGSH':0.20}

r_31, d_31 = simulate_rebalance(ret_long, V5C_3_1)
r_31f, d_31f = simulate_rebalance(ret_long, V5C_3_1f)

show_compare([
    metrics(r_31, d_31, '3.1 (VOO15 QQQ15)'),
    metrics(r_31f, d_31f, '3.1f (VOO10 QQQ20)'),
])

events_long = {
    '2008 GFC':       ('2007-10-09', '2009-03-09'),
    '2008 急跌':       ('2008-09-01', '2008-12-31'),
    '2018-Q4':         ('2018-10-01', '2018-12-31'),
    '2020 COVID':     ('2020-02-19', '2020-04-30'),
    '2020-2021 反弹':  ('2020-04-30', '2021-12-31'),
    '2022 Bear':      ('2022-01-01', '2022-12-31'),
    '2023-2024 复苏':  ('2023-01-01', '2024-12-31'),
    '2025 Q1 关税':    ('2025-01-01', '2025-04-30'),
}
print('\n关键时期:')
print(f"{'时期':<22} {'3.1':>10} {'3.1f':>10} {'Δ':>10}")
for n, (s, e) in events_long.items():
    if pd.Timestamp(s) < r_31.index[0]: continue
    a = (1 + r_31.loc[s:e]).prod() - 1
    b = (1 + r_31f.loc[s:e]).prod() - 1
    print(f'{n:<22} {a:>+9.2%}  {b:>+9.2%}  {b-a:>+9.2%}')

In [ ]:
# ============================================================
# 测试 2: 16Y - V5C 3.1 vs V5C 3.1f vs V5C 3.1g (含 TQQQ)
# ============================================================
print('=' * 90)
print('测试 2: 16Y - V5C 3.1 vs V5C 3.1f vs V5C 3.1g (TQQQ 5%)')
print('=' * 90)

data_16y = data[['VOO','QQQ','TQQQ','HQH','XLV','VGSH','GLDM','BCX']].dropna()
ret_16y = data_16y.pct_change().dropna()
print(f'窗口: {data_16y.index[0].date()} → {data_16y.index[-1].date()} ({len(data_16y)/252:.1f} 年)')

V5C_3_1g = {'VOO':0.10,'QQQ':0.15,'TQQQ':0.05,'HQH':0.10,'XLV':0.10,'GLDM':0.20,'BCX':0.10,'VGSH':0.20}

r_31_16, d_31_16 = simulate_rebalance(ret_16y, V5C_3_1)
r_31f_16, d_31f_16 = simulate_rebalance(ret_16y, V5C_3_1f)
r_31g_16, d_31g_16 = simulate_rebalance(ret_16y, V5C_3_1g)

show_compare([
    metrics(r_31_16, d_31_16, '3.1 (基准)'),
    metrics(r_31f_16, d_31f_16, '3.1f (+5% QQQ)'),
    metrics(r_31g_16, d_31g_16, '3.1g (+5% TQQQ)'),
])

events_16y = {
    '2018-Q4 跌势':       ('2018-10-01', '2018-12-31'),
    '2020 COVID':         ('2020-02-19', '2020-04-30'),
    '2020 流动性极端':     ('2020-03-09', '2020-03-23'),
    '2020-2021 科技牛':    ('2020-04-30', '2021-12-31'),
    '2022 Bear':         ('2022-01-01', '2022-12-31'),
    '2022 NASDAQ 极端':   ('2022-01-01', '2022-10-12'),
    '2023-2024 AI 牛':    ('2023-01-01', '2024-12-31'),
    '2025 Q1 关税':       ('2025-01-01', '2025-04-30'),
}
print('\n关键时期 (16Y 窗口):')
print(f"{'时期':<22} {'3.1':>10} {'3.1f':>10} {'3.1g':>10}")
for n, (s, e) in events_16y.items():
    a = (1 + r_31_16.loc[s:e]).prod() - 1
    b = (1 + r_31f_16.loc[s:e]).prod() - 1
    c = (1 + r_31g_16.loc[s:e]).prod() - 1
    print(f'{n:<22} {a:>+9.2%}  {b:>+9.2%}  {c:>+9.2%}')

In [ ]:
# ============================================================
# 单标的特征 (VOO vs QQQ vs TQQQ)
# ============================================================
print('=' * 70)
print('单标的特征 (23.8Y for VOO/QQQ; 16Y for TQQQ)')
print('=' * 70)

for label, ser in [('VOO (23.8Y)', ret_long['VOO']),
                    ('QQQ (23.8Y)', ret_long['QQQ']),
                    ('TQQQ (16Y)', ret_16y['TQQQ'])]:
    cum = (1+ser).cumprod()
    n = len(ser)
    cagr = cum.iloc[-1]**(252/n) - 1
    vol = ser.std()*np.sqrt(252)
    sharpe = (cagr-0.04)/vol
    rm = cum.expanding().max()
    dd = (cum/rm - 1).min()
    print(f'{label:<14}: CAGR {cagr:>+7.2%}  Vol {vol:>5.2%}  Sharpe {sharpe:>+.3f}  MaxDD {dd:>+7.2%}')

print('\n危机时期单标的:')
events_check = list(events_long.keys())
print(f"{'时期':<22} {'VOO':>10} {'QQQ':>10} {'TQQQ':>10}")
for n, (s, e) in events_long.items():
    if pd.Timestamp(s) < ret_long.index[0]: continue
    voo_r = (1 + ret_long['VOO'].loc[s:e]).prod() - 1
    qqq_r = (1 + ret_long['QQQ'].loc[s:e]).prod() - 1
    if pd.Timestamp(s) < ret_16y.index[0]:
        tqqq_str = '   N/A'
    else:
        tqqq_r = (1 + ret_16y['TQQQ'].loc[s:e]).prod() - 1
        tqqq_str = f'{tqqq_r:>+9.2%}'
    print(f'{n:<22} {voo_r:>+9.2%}  {qqq_r:>+9.2%}  {tqqq_str}')

In [ ]:
# ============================================================
# 净值 + 回撤可视化 (16Y 窗口)
# ============================================================
cum_31 = (1 + r_31_16).cumprod()
cum_31f = (1 + r_31f_16).cumprod()
cum_31g = (1 + r_31g_16).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
axes[0].plot(cum_31, label='V5C 3.1 基准 (VOO15+QQQ15)', linewidth=2, alpha=0.85)
axes[0].plot(cum_31f, label='V5C 3.1f (VOO10+QQQ20)', linewidth=2, alpha=0.85)
axes[0].plot(cum_31g, label='V5C 3.1g (VOO10+QQQ15+TQQQ5)', linewidth=2, alpha=0.85)
axes[0].set_title('净值对比 (16Y, log scale)', fontsize=14)
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(alpha=0.3)

for cum, label, color in [(cum_31, '3.1', 'C0'), (cum_31f, '3.1f', 'C1'), (cum_31g, '3.1g', 'C2')]:
    rm = cum.expanding().max()
    dd = (cum / rm) - 1
    axes[1].fill_between(dd.index, dd.values, 0, alpha=0.3, color=color, label=label)
axes[1].set_title('回撤对比', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 决策标准

### 接受 V5C 3.1f (VOO 10 + QQQ 20)
- 23.8Y Sharpe ≥ V5C 3.1
- 2008 GFC Max DD 不显著恶化 (< 3pp)
- 2022 Bear 损失增量 < 5pp

### 接受 V5C 3.1g (TQQQ 5%)
- 16Y Sharpe **显著高于** V5C 3.1 (>0.05)
- 2022 Bear 期间 Max DD 加深 < 8pp (杠杆代价可控)
- TQQQ 5% 没有触发危险回撤 (>50% 时整体仓位不超 -30%)

### 保持 V5C 3.1
- 任何方案 Sharpe 不显著优于基准
- 或 Max DD 显著恶化 (说明只是 beta 不是 alpha)

## 提醒

**这是 V5C 3.1 设计期内的最后调整 (5/4 截止)**.
如果 3.1f 或 3.1g 显著优胜, 可考虑修订到 V5C 3.2.
如果都不优胜, 维持 V5C 3.1 不变, 真正进入 5 月沉淀.